# Download Data From MediaWiki

In [ ]:
import os
import requests
import bz2
from tqdm import tqdm

In [ ]:
urls = [
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwiktionary/2026-04-01/xml/bzip2/enwiktionary-2026-04-01-p7p11575490.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwikiquote/2026-04-01/xml/bzip2/enwikiquote-2026-04-01-p1p305881.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwikibooks/2026-04-01/xml/bzip2/enwikibooks-2026-04-01-p1p482395.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/simplewiki/2026-04-01/xml/bzip2/simplewiki-2026-04-01-p1p1261009.xml.bz2",                    
]

In [ ]:
# Output folders
download_dir = "/kaggle/working/downloads"
extract_dir = "/kaggle/working/extracted"

os.makedirs(download_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)


def download_file(url, output_path):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(output_path, 'wb') as file, tqdm(
        desc=os.path.basename(output_path),
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)
                bar.update(len(chunk))


def extract_bz2(input_path, output_path):
    with bz2.open(input_path, 'rb') as f_in, open(output_path, 'wb') as f_out:
        for data in iter(lambda: f_in.read(1024 * 1024), b''):
            f_out.write(data)


# Process each URL
for url in urls:
    filename = os.path.basename(url)
    bz2_path = os.path.join(download_dir, filename)

    xml_filename = filename.replace(".bz2", "")
    xml_path = os.path.join(extract_dir, xml_filename)

    print(f"\nDownloading: {filename}")
    download_file(url, bz2_path)

    print(f"Extracting: {xml_filename}")
    extract_bz2(bz2_path, xml_path)

    print(f"Done: {xml_filename}")

print("\nAll files downloaded and extracted successfully!")